In [1]:
import pandas as pd
import numpy as np

np.random.seed(505)

n = 1400

patient_id = [f"P{200000+i}" for i in range(n)]
hospital = np.random.choice(
    ["Hospital_A", "Hospital_B", "Hospital_C"],
    size=n,
    p=[0.35, 0.4, 0.25]
)

age = np.round(np.clip(np.random.normal(59, 18, n), 18, 95)).astype(int)

# Case severity differs by hospital
severity = []
for h in hospital:
    if h == "Hospital_A":
        severity.append(np.random.choice(["low", "medium", "high"], p=[0.55, 0.32, 0.13]))
    elif h == "Hospital_B":
        severity.append(np.random.choice(["low", "medium", "high"], p=[0.28, 0.42, 0.30]))
    else:
        severity.append(np.random.choice(["low", "medium", "high"], p=[0.42, 0.38, 0.20]))
severity = np.array(severity)

comorbidity_score = np.where(
    severity == "low",
    np.random.poisson(1.2, n),
    np.where(severity == "medium",
             np.random.poisson(2.6, n),
             np.random.poisson(4.3, n))
)

treatment_type = np.where(
    severity == "high",
    np.random.choice(["standard", "advanced"], size=n, p=[0.35, 0.65]),
    np.where(severity == "medium",
             np.random.choice(["standard", "advanced"], size=n, p=[0.6, 0.4]),
             np.random.choice(["standard", "advanced"], size=n, p=[0.82, 0.18]))
)

# Base recovery probability depends strongly on severity and comorbidities
base_recovery = np.where(
    severity == "low", 0.94,
    np.where(severity == "medium", 0.78, 0.54)
)

# Small hospital-specific quality effect
hospital_effect = np.where(
    hospital == "Hospital_A", -0.01,
    np.where(hospital == "Hospital_B", 0.03, 0.01)
)

treatment_effect = np.where(treatment_type == "advanced", 0.03, 0.0)

recovery_probability = base_recovery + hospital_effect + treatment_effect - 0.025 * comorbidity_score
recovery_probability = np.clip(recovery_probability, 0.05, 0.99)

recovered = np.random.binomial(1, recovery_probability, n)

length_of_stay = np.where(
    severity == "low",
    np.random.normal(2.8, 1.0, n),
    np.where(severity == "medium",
             np.random.normal(5.5, 1.8, n),
             np.random.normal(9.5, 3.0, n))
)
length_of_stay = np.round(np.clip(length_of_stay + 0.3 * comorbidity_score, 1, None), 1)

readmission_probability = np.where(
    severity == "low", 0.04,
    np.where(severity == "medium", 0.1, 0.2)
) + np.where(recovered == 1, -0.02, 0.06)
readmission_probability = np.clip(readmission_probability, 0.01, 0.95)

readmitted_30d = np.random.binomial(1, readmission_probability, n)

df = pd.DataFrame({
    "patient_id": patient_id,
    "hospital": hospital,
    "age": age,
    "severity": severity,
    "comorbidity_score": comorbidity_score,
    "treatment_type": treatment_type,
    "recovered": recovered,
    "length_of_stay_days": length_of_stay,
    "readmitted_30d": readmitted_30d
})

df.to_csv("topic_D2_hospital_outcomes_raw.csv", index=False)
print("Saved: topic_D2_hospital_outcomes_raw.csv")
print(df.head())
print(df.shape)

Saved: topic_D2_hospital_outcomes_raw.csv
  patient_id    hospital  age severity  comorbidity_score treatment_type  \
0    P200000  Hospital_C   78      low                  3       standard   
1    P200001  Hospital_A   69      low                  3       standard   
2    P200002  Hospital_A   70      low                  2       standard   
3    P200003  Hospital_A   87      low                  0       standard   
4    P200004  Hospital_B   32   medium                  5       standard   

   recovered  length_of_stay_days  readmitted_30d  
0          1                  3.8               0  
1          1                  5.1               0  
2          1                  2.6               0  
3          1                  4.1               0  
4          0                  7.9               0  
(1400, 9)
